## [ 온도(ta)로 계절 군집 설정한 'season' 변수 만들기 ]
- 컨셉 : 각 지점에 맞는 정확한 여름기간을 계산하여 레이블링
- 여름은 0, 겨울(열수요가 증가하는 나머지 계절)은 1
- 여름 시작일 : 9일 동안 일 평균기온이 20°C 이상 올라간 후, 다시 떨어지지 않을 때, 그 첫번째 날
- 여름 종료일 : 9일동안 일 평균기온이 20°C 미만 내려간 후, 다시 올라가지 않을 때, 그 첫번째 날

In [ ]:
################ 사용 전 주의사항 ##################
# 체감온도(ta_chi)에 비해 온도(ta)는 결측치가 매우 많음, 그리고 모든 데이터는 군집이 지정되어야 함.
# 그런 이유로 지점별로 파티션화 하여, 선형 보간을 실행함
# 결론: 3번 코드는 필수적으로 사용하는 것을 추천

In [50]:
# 1번 : 기본적인 전처리 코드 (필요 시 사용)
# 데이터 로드
import pandas as pd
import numpy as np
df = pd.read_csv('./train_heat.csv')

# 첫번째 column 제거
df = df.iloc[:, 1:]

# column 이름 예쁘게 정리
df.columns = df.columns.str.replace('train_heat.', '', regex=False)

# 미관측치는 NA로 대체
df.replace(-99, np.nan, inplace=True)
df['wd'] = df['wd'].replace(-9.9, np.nan) # 풍속 결측치

In [52]:
# 2번 : 사회 변수 추가 (필요 시 사용)
df['datetime'] = pd.to_datetime(df['tm'].astype(str), format='%Y%m%d%H')
df['weekday'] = df['datetime'].dt.weekday # 월요일이 0
df['hour'] = df['datetime'].dt.hour

# 1) 주말 변수
df['is_weekend'] = df['weekday'] >= 5
# 2) 출퇴근 변수
df['work_time'] = 0
target_data = [7, 8, 9, 19, 20, 21, 22]
df.loc[df['hour'].isin(target_data), 'work_time'] = 1

In [ ]:
# 3번 : 지점별로 파티션 나눈 뒤 선형 보간 (사용 추천)
# 1. 'branch_id'와 'datetime'을 기준으로 데이터 정렬 
df = df.sort_values(by=['branch_id', 'datetime']).reset_index(drop=True)

# 2. 'branch_id'별로 그룹화하여 연속형 변수 보간 적용
continuous_vars = ['ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi','heat_demand']
for col in continuous_vars:
    df[col] = df.groupby('branch_id')[col].transform(lambda group: group.interpolate(method='linear', limit_direction='both'))

# 모든 NaN 값이 채워졌는지 확인 (선택 사항)
print("\n--- 보간 후 각 변수의 NaN 개수 ---")
print(df[continuous_vars].isnull().sum())

### season 레이블 지정

In [ ]:
################ 함수 구간  ####################

In [56]:
# 여름 기간 찾기
def avg_temp(df):
    df['date'] = df['datetime'].dt.date
    df['date'] = pd.to_datetime(df['date'])
    
    # branch_id와 date로 그룹화하여 일 평균 기온 계산
    df = df.groupby(['branch_id', 'date'])['ta'].mean().reset_index()
    df.rename(columns={'ta': 'avg_temp'}, inplace=True)
    
    # 9일 이동 평균 계산 (각 지점별로 독립적으로 계산)
    df['rolling_avg_9day'] = df.groupby('branch_id')['avg_temp'].transform(lambda x: x.rolling(window=9, center=True).mean())
    
    # 이동 평균 계산 시 생기는 NaN 값 처리 (분석 범위에서 제외)
    df = df.dropna(subset=['rolling_avg_9day']).reset_index(drop=True)

    return df

def find_summer_start(df_single_branch):

    summer_start_date = None
    
    # 9일 연속 조건을 확인하므로, 배열의 끝에서 8일 이전까지만 순회
    for i in range(len(df_single_branch) - 8): 
        current_date = df_single_branch.loc[i, 'date']
        
        # 1. 현재 날짜의 9일 이동 평균이 20°C 이상인지 확인
        if df_single_branch.loc[i, 'rolling_avg_9day'] >= 20:
            # 2. 이 날부터 연속해서 9일 동안 9일 이동 평균이 20°C 이상을 유지하는지 확인
            is_consistent_for_9_days = True
            for k in range(i, i + 9): # 현재 날짜(i)부터 8일 뒤(i+8)까지, 총 9일
                if df_single_branch.loc[k, 'rolling_avg_9day'] < 20:
                    is_consistent_for_9_days = False
                    break # 20°C 미만으로 떨어지면 9일 연속 조건 불만족
            
            # 3. 9일 연속 조건이 만족되면 해당 날짜를 여름 시작일로 설정하고 종료
            if is_consistent_for_9_days:
                summer_start_date = current_date
                break 
                
    return summer_start_date
    
def find_summer_end(df_single_branch, summer_start_date=None):
    summer_end_date = None

    start_idx = 0
    if summer_start_date is not None and not df_single_branch[df_single_branch['date'] == summer_start_date].empty:
        start_idx = df_single_branch[df_single_branch['date'] == summer_start_date].index[0]

    # 여름 시작일 이후부터 탐색
    for i in range(start_idx, len(df_single_branch) - 8): # 9일 연속 확인을 위해 범위 조정
        current_date = df_single_branch.loc[i, 'date']

        # 현재 날짜의 9일 이동 평균이 20°C 미만인지 확인
        if df_single_branch.loc[i, 'rolling_avg_9day'] < 20:
            # 이 날부터 연속해서 9일 동안 9일 이동 평균이 20°C 미만을 유지하는지 확인
            is_consistent_for_9_days = True
            for k in range(i, i + 9):
                if df_single_branch.loc[k, 'rolling_avg_9day'] >= 20:
                    is_consistent_for_9_days = False
                    break

            # 9일 연속 조건이 만족되면 해당 날짜를 여름 종료일로 설정하고 종료
            if is_consistent_for_9_days:
                summer_end_date = current_date
                break
    return summer_end_date
    
# 각 지점별로 여름 기간 찾기
def find_summer_by_branch(df):
    summer_periods_by_branch = {}
    df_original = df.copy()
    df = avg_temp(df)
    unique_branches = df['branch_id'].unique()
    
    for branch_id in unique_branches:
        # 해당 지점의 데이터만 필터링
        df_single_branch = df[df['branch_id'] == branch_id].copy().reset_index(drop=True)
        
        # 여름 시작일 찾기
        summer_start = find_summer_start(df_single_branch)
        
        # 여름 종료일 찾기 (찾은 여름 시작일 전달)
        summer_end = find_summer_end(df_single_branch, summer_start_date=summer_start)
        
        if summer_start and summer_end:
            summer_periods_by_branch[branch_id] = {'start': summer_start, 'end': summer_end}
        elif summer_start:
            summer_periods_by_branch[branch_id] = {'start': summer_start, 'end': None}
        elif summer_end:
            summer_periods_by_branch[branch_id] = {'start': None, 'end': summer_end}
        else:
            summer_periods_by_branch[branch_id] = {'start': None, 'end': None}
    
    # 최종 결과 딕셔너리에 저장
    print("\n--- 모든 지점의 여름 기간 요약 ---")
    for branch, period in summer_periods_by_branch.items():
        start_str = period['start'].strftime('%Y-%m-%d') if period['start'] else 'N/A'
        end_str = period['end'].strftime('%Y-%m-%d') if period['end'] else 'N/A'
        print(f"지점 {branch}: 시작일 {start_str}, 종료일 {end_str}")

    # season 열 추가는 원본 데이터에
    df_original['date'] = df_original['datetime'].dt.date
    df_original['date'] = pd.to_datetime(df_original['date'])
    df_original['season'] = 1
    
    # 여름 기간 레이블링
    for branch_id, period_info in summer_periods_by_branch.items():
        start_date = period_info['start']
        end_date = period_info['end']
        
        if start_date and end_date:
            df_original.loc[
                (df_original['branch_id'] == branch_id) &
                (df_original['date'] >= start_date) &
                (df_original['date'] <= end_date),
                'season'
            ] = 0

    df_original.head()
    return df_original

In [ ]:
###### 함수 사용 구간 ########

In [58]:
# 1. 연도 별로 데이터 분리
df_2021 = df[(df['datetime'] >= '2021-01-01') & (df['datetime'] < '2022-01-01')]
df_2022 = df[(df['datetime'] >= '2022-01-01') & (df['datetime'] < '2023-01-01')]
df_2023 = df[df['datetime'] >= '2023-01-01']
df_list = [df_2021, df_2022, df_2023]

In [ ]:
# 2. 연도별로 지점별 여름 기간 계산 및 저장
for idx, df in enumerate(df_list):
    df_list[idx] = find_summer_by_branch(df)
    
df_2021, df_2022, df_2023 = df_list

In [ ]:
# 3. 2021 ~ 2023년 데이터 병합
df_cleaned = pd.concat(df_list, ignore_index=True)
df_cleaned.head()
# df_cleaned.to_csv('train_cleaned.csv', index=False) # 필요시 사용